# 函数与参数

学习目标：能定义与调用函数，正确传递参数、设计回调和递归，并理解标签模板的输入。

前置知识：变量、对象引用、条件循环、字符串模板和数组基本索引。

适用版本：ECMAScript 2025（ECMA-262 第 16 版）、Node.js 24.11.0；.mjs 文件按 ES 模块运行并采用严格模式。console 是宿主输出 API。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/07-functions-and-parameters/。

1. [main.mjs](scripts/07-functions-and-parameters/main.mjs)：按正文顺序运行全部正常示例。
2. [default-parameter-tdz.mjs](scripts/07-functions-and-parameters/default-parameter-tdz.mjs)：默认参数从左到右初始化，不能先读取尚未初始化的后一个参数。
3. [arrow-constructor.mjs](scripts/07-functions-and-parameters/arrow-constructor.mjs)：箭头函数不能作为构造函数调用。

Step 1：从项目根目录进入本章工作目录。

```bash
cd content/编程语言/javascript
```

Step 2：运行全部正常示例，按各片段中的输出注释核对。

```bash
node scripts/07-functions-and-parameters/main.mjs
```

下文正常片段依次对应 main.mjs 中的代码；每段给出自身输入与定义。错误文件仅在相应小节单独运行。

## 1 声明、表达式、调用与返回

函数声明用 function、函数名、参数列表和函数体定义可复用操作；函数表达式在表达式位置创建函数值，可以赋给变量。定义函数不执行函数体，函数名后加圆括号才调用；参数是定义中的名称，实参是调用时传入的值。

return 结束本次调用并返回值，不带表达式或执行到函数末尾时返回 undefined。return 与表达式间不能随意换行，否则自动分号插入可能让它提前结束。函数声明的提前可用性和函数表达式变量的初始化属于作用域问题，下一章展开。

```javascript
function rectangleArea(width, height) {
  return width * height;
}
const add = function (left, right) {
  return left + right;
};
function finish() {
  return;
}
function noReturn() {
  const result = 4;
}
console.log(rectangleArea(4, 3), add(4, 3), finish(), noReturn());
function lineBreak() {
  return
  10;
}
console.log(lineBreak());
// 输出依次为：
// 12 7 undefined undefined
// undefined
```

## 2 默认参数与缺少实参

没有对应实参的参数初值为 undefined。默认参数仅在实参缺失或为 undefined 时生效，null、0 和空字符串不会触发默认值。默认表达式在每次调用需要时求值，并按参数从左到右初始化；后面的默认值可以使用前面已初始化的参数，反向引用可能进入暂时性死区。

有默认表达式时，函数体内的局部声明不能被默认表达式当作其内层可见变量。下面把一个默认对象就地创建，说明不同调用不会自动共享这个新对象。

```javascript
function total(price, quantity = 1, subtotal = price * quantity) {
  return subtotal;
}
console.log(total(8), total(8, undefined), total(8, null), total(8, 0));
function fresh(options = { count: 0 }) {
  options.count += 1;
  return options.count;
}
console.log(fresh(), fresh());
function missing(value) {
  return value;
}
console.log(missing());
// 输出依次为：
// 8 8 0 0
// 1 1
// undefined
```

## 3 剩余参数与实参展开

定义参数时，...values 是剩余参数（rest parameter），把尚未匹配的实参收集为真正的数组；values 是本例自定名称。剩余参数必须放最后，不能带默认值。

调用处的 ...numbers 是展开语法（spread syntax），从可迭代对象逐项提供实参，与参数收集方向相反；... 在这里是语法，不是省略内容的标记。不要将来源不受控的超大数组一次性展开为实参，引擎对实参数量存在实际限制，适合改用循环累计。

```javascript
function sum(first = 0, ...others) {
  let result = first;
  for (const value of others) result += value;
  return result;
}
console.log(sum(), sum(2, 3, 4));
const numbers = [2, 3, 4];
console.log(sum(...numbers), Array.isArray(numbers));
function restKind(...values) {
  return Array.isArray(values);
}
console.log(restKind(1, 2));
// 输出依次为：
// 0 9
// 9 true
// true
```

## 4 arguments 与箭头函数

普通函数有自己的 arguments 类数组对象：可以按索引读取实际实参，并通过 length 查看数量，但它本身不是 Array。当前 ES 模块使用严格模式，arguments 索引与命名参数不保持赋值联动；非严格模式的简单参数列表有历史映射规则，不能套用到这里。新接口通常用剩余参数表达可变参数。

箭头函数用 => 定义。表达式函数体隐式返回表达式值；花括号函数体需要显式 return。直接返回对象字面量时用括号包住。箭头函数没有自己的 arguments 或 this，也不能用 new 构造；它会从词法外层取得这些绑定。this 的调用差异在专章展开。

```javascript
function inspect(first) {
  first = 99;
  return `${first}:${arguments[0]}:${arguments.length}:${Array.isArray(arguments)}`;
}
console.log(inspect(4, 5));
const double = value => value * 2;
const makeRecord = value => ({ value });
const blockBody = value => { value * 2; };
console.log(double(5), makeRecord(3).value, blockBody(5));
function outer(value) {
  const readOuterArguments = () => arguments[0];
  return readOuterArguments(99);
}
console.log(outer("外层实参"));
// 输出依次为：
// 99:4:2:false
// 10 3 undefined
// 外层实参
```

## 5 按值传递与共享对象

参数按值传递。传入原始值时，参数重新赋值不会改变调用者变量；传入对象时，参数取得同一个对象的引用值，修改属性能被调用者观察到，但把参数改为另一个对象不会改写调用者的绑定。

因此“函数改了对象”与“函数换掉了参数保存的对象”是两件事。接口应明确是否允许修改输入；不希望共享修改时，可在对象章节学习复制策略。

```javascript
function revise(number, item) {
  number = 100;
  item.score = 90;
  item = { score: 0 };
  return `${number}:${item.score}`;
}
const originalNumber = 5;
const report = { score: 80 };
console.log(revise(originalNumber, report));
console.log(originalNumber, report.score);
// 输出依次为：
// 100:0
// 5 90
```

## 6 回调与立即调用函数表达式

把函数作为实参交给其他函数，在适当位置由接收方调用，称为回调（callback）。传递函数值时不加调用括号；是否同步、何时执行、返回值是否被使用，都由接收方规定。回调不天然意味着异步。

立即调用函数表达式（immediately invoked function expression，IIFE）将函数表达式包在括号中并立即调用，可建立一次性的局部作用域并返回结果。ES 模块和块作用域已经解决了许多隔离需求，不必把所有代码都包成 IIFE。

```javascript
function applyTwice(value, transform) {
  return transform(transform(value));
}
function increment(value) {
  return value + 1;
}
console.log(applyTwice(3, increment));
const configured = (function (prefix) {
  const suffix = "ready";
  return `${prefix}:${suffix}`;
})("job");
console.log(configured);
// 输出依次为：
// 5
// job:ready
```

## 7 递归与终止条件

递归是函数调用自身。每次调用要处理更小的问题，并在基线条件直接返回；否则无法结束。这里 factorial 约定只接收 0–10 的整数，先拒绝范围外输入，再把 n 缩减为 n - 1；n 表示非负整数因子上限，0 的阶乘约定为 1。

函数调用需要执行资源，大深度递归可能触发宿主的调用栈限制。不要依赖某个固定递归深度，也不要因为写在 return 后面就假定运行环境一定消除调用栈。异常语法这里只用于拒绝无效输入，异常处理在对应章节展开。

```javascript
function factorial(n) {
  if (!Number.isInteger(n) || n < 0 || n > 10) {
    throw new RangeError("n 必须是 0 到 10 的整数");
  }
  if (n === 0) return 1;
  return n * factorial(n - 1);
}
console.log(factorial(0), factorial(5));
// 输出依次为：
// 1 120
```

## 8 标签模板的调用输入

在模板前放置标签函数（tag function），形成标签模板（tagged template）调用。第一个参数是模板的文本片段数组，后续参数是各个插值表达式的原始结果；有几个插值，就比它多一个文本片段，哪怕首尾是空字符串。标签可以返回任意值，不一定是字符串。

本例 parts 是片段数组，values 收集插值结果。parts.raw 保存未处理转义的片段；同一模板位置再次求值会使用同一个冻结的模板对象。String.raw 就利用原始片段组成字符串。标签机制本身不保证 HTML 转义或 SQL 参数化，安全性取决于标签函数的具体实现。

```javascript
function inspectTemplate(parts, ...values) {
  return `${parts.length}:${values.length}:${typeof values[0]}:${parts[0].length}:${parts.raw[0].length}`;
}
console.log(inspectTemplate`A\n${3}!`);
function numberOnly(parts, value) {
  return value * 2;
}
console.log(numberOnly`amount=${6}`);
// 输出依次为：
// 2:1:number:2:3
// 12
```

## 9 独立观察错误边界

以下文件分别启动新进程；预期退出码为 1。先根据代码判断错误原因，再运行相应命令，核对错误名称及对应位置。错误消息全文由宿主决定。

默认参数从左到右初始化，不能先读取尚未初始化的后一个参数。

```javascript
function read(first = second, second = 2) {
  return first;
}
console.log(read());
// 预期错误：ReferenceError；Cannot access 'second' before initialization
```

Step 1：独立运行 scripts/07-functions-and-parameters/default-parameter-tdz.mjs。

```bash
node scripts/07-functions-and-parameters/default-parameter-tdz.mjs
```

箭头函数不能作为构造函数调用。

```javascript
const build = () => ({ ready: true });
console.log(new build());
// 预期错误：TypeError；build is not a constructor
```

Step 2：独立运行 scripts/07-functions-and-parameters/arrow-constructor.mjs。

```bash
node scripts/07-functions-and-parameters/arrow-constructor.mjs
```

## 本章小结

- 函数值、调用结果和返回值是三个不同位置上的概念；回调不等于异步。
- 默认参数处理 undefined，剩余参数收集实参，展开在调用处展开可迭代值。
- 参数重新赋值不改调用者绑定；修改共享对象则可能形成可见副作用。
- 递归要有终止条件；标签模板让函数分别接收文本片段与插值值。

## 练习

1. 实现 quantity 默认是 1 的计价函数，用 undefined、null、0 测试；核对只有 undefined 触发默认值。
2. 将三个整数通过剩余参数求和，再使用数组展开调用；核对两次结果相同，空输入返回 0。
3. 修改 revise，使它返回新对象并保留输入对象的 score 为 80；同时验证返回对象与输入对象身份不同。
4. 实现接收两个插值的标签函数，返回插值和；用 2、5 核对得到数值 7，并检查片段数组长度为 3。

## 参考与引用来源

- TC39（tc39.es）：[§15.1–15.3 参数、普通函数与箭头函数](https://tc39.es/ecma262/2025/multipage/ecmascript-language-functions-and-classes.html#sec-function-definitions)、[§14.10 return](https://tc39.es/ecma262/2025/multipage/ecmascript-language-statements-and-declarations.html#sec-return-statement)、[§13.3.8 实参展开](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-argument-lists)、[§13.3.11 标签调用](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-tagged-templates)、[§13.2.8.4 模板对象](https://tc39.es/ecma262/2025/multipage/ecmascript-language-expressions.html#sec-gettemplateobject)、[§22.1.2.4 String.raw](https://tc39.es/ecma262/2025/multipage/text-processing.html#sec-string.raw)、[§10.2.11 参数、arguments 和声明实例化](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-functiondeclarationinstantiation)、[§10.4.4 arguments 对象与映射条件](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-arguments-exotic-objects)：ECMAScript 2025 的函数调用、参数、返回、箭头及标签模板规则。
- MDN：[Functions](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Functions)：按值传递、回调、递归和参数教学用法对照。